To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Cài đặt wandb để theo dõi training
%pip install -qqq wandb

: 

### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [ ]:
from unsloth import FastLanguageModel
import torch

# ✅ TỐI ƯU CHO 16GB VRAM:
# - Sử dụng BNB 4-bit quantization (tiết kiệm VRAM hơn MXFP4)
# - max_seq_length = 2048 (đủ cho dữ liệu pháp luật, không quá lớn)
# - load_in_4bit = True (bắt buộc cho 16GB VRAM)

max_seq_length = 2048  # Đủ cho hầu hết câu trả lời pháp luật
dtype = None  # Auto detect

# ✅ QUAN TRỌNG: Sử dụng BNB 4-bit model để tiết kiệm VRAM tối đa
# unsloth/gpt-oss-20b-unsloth-bnb-4bit: BitsAndBytes 4-bit (tiết kiệm VRAM nhất)
# unsloth/gpt-oss-20b: MXFP4 format (nhanh hơn nhưng tốn VRAM hơn)
model_name = "unsloth/gpt-oss-20b-unsloth-bnb-4bit"  # ✅ Tối ưu cho 16GB VRAM

print(f"🦥 Đang load model GPT-OSS 20B với cấu hình tối ưu cho 16GB VRAM...")
print(f"   Model: {model_name}")
print(f"   Max sequence length: {max_seq_length}")
print(f"   Quantization: 4-bit (BNB)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,  # ✅ BNB 4-bit model
    dtype = dtype,  # None for auto detection
    max_seq_length = max_seq_length,
    load_in_4bit = True,  # ✅ Bắt buộc: 4-bit quantization để fit vào 16GB VRAM
    full_finetuning = False,  # Chỉ train LoRA adapters (không train full model)
    # token = "hf_...", # use one if using gated models
)

# Kiểm tra VRAM sau khi load model
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated_memory = torch.cuda.memory_allocated() / 1024**3
    reserved_memory = torch.cuda.memory_reserved() / 1024**3
    print(f"\n✓ Đã load model thành công!")
    print(f"   GPU Memory: {gpu_memory:.2f} GB")
    print(f"   Đã sử dụng: {reserved_memory:.2f} GB ({reserved_memory/gpu_memory*100:.1f}%)")
    print(f"   Còn lại: {gpu_memory - reserved_memory:.2f} GB")
    if reserved_memory > 14:
        print(f"   ⚠️ Cảnh báo: VRAM sử dụng cao ({reserved_memory:.2f}GB), có thể cần giảm max_seq_length")
else:
    print(f"\n✓ Đã load model với max_seq_length = {max_seq_length}")

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [ ]:
# ✅ CẤU HÌNH LoRA TỐI ƯU CHO 16GB VRAM:
# - r=8: Rank nhỏ để tiết kiệm VRAM (có thể tăng lên 16 nếu còn VRAM)
# - use_gradient_checkpointing="unsloth": Tiết kiệm 30% VRAM
# - lora_dropout=0: Tối ưu tốc độ và VRAM

print("🔧 Đang thiết lập LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,  # ✅ Rank nhỏ để tiết kiệm VRAM (8 cho 16GB, có thể tăng lên 16 nếu còn VRAM)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],  # ✅ Tất cả attention và MLP layers
    lora_alpha = 16,  # Alpha = 2 * r (khuyến nghị)
    lora_dropout = 0,  # ✅ = 0 để tối ưu VRAM và tốc độ
    bias = "none",  # ✅ Không train bias để tiết kiệm VRAM
    # ✅ QUAN TRỌNG: "unsloth" gradient checkpointing tiết kiệm 30% VRAM
    use_gradient_checkpointing = "unsloth",  # True hoặc "unsloth" (unsloth tốt hơn)
    random_state = 3407,  # Reproducibility
    use_rslora = False,  # Rank stabilized LoRA (không cần cho 16GB)
    loftq_config = None,  # LoftQ quantization (không cần vì đã dùng 4-bit)
)

# Kiểm tra số lượng parameters có thể train
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ LoRA adapters đã được thiết lập!")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)")
print(f"   Total parameters: {total_params:,}")

# Kiểm tra VRAM sau khi setup LoRA
if torch.cuda.is_available():
    reserved_memory = torch.cuda.memory_reserved() / 1024**3
    print(f"   VRAM sau LoRA setup: {reserved_memory:.2f} GB")

### 📊 Cấu hình Weights & Biases (wandb)

Wandb giúp theo dõi training metrics, loss curves, và so sánh các runs khác nhau. 

Để sử dụng wandb:
1. **Login lần đầu**: Bỏ comment dòng `wandb.login()` và chạy (cần API key từ https://wandb.ai)
2. **Hoặc dùng offline mode**: Chỉ lưu logs local không upload lên cloud


In [ ]:
import wandb
import os

# Chọn 1 trong 2 options sau:

# OPTION 1: Login vào wandb (cần API key lần đầu)
# Lấy API key tại: https://wandb.ai/authorize
# wandb.login()

# OPTION 2: Dùng offline mode (không cần login, chỉ lưu local)
os.environ["WANDB_MODE"] = "offline"

# Khởi tạo wandb project
wandb.init(
    project="gpt-oss-legal-vietnamese",  # Tên project trên wandb
    name="gpt-oss-20b-16gb-vram",         # Tên run này (đã cập nhật)
    config={
        "model": "unsloth/gpt-oss-20b-unsloth-bnb-4bit",  # ✅ Đã cập nhật model name
        "max_seq_length": 2048,
        "lora_r": 8,
        "lora_alpha": 16,
        "gradient_accumulation_steps": 8,  # ✅ Đã cập nhật
        "per_device_batch_size": 1,  # ✅ Đã cập nhật
        "optimizer": "adamw_8bit",  # ✅ Đã cập nhật
        "dataset": "qaset_article + all_chunks_final",
        "vram_target": "16GB",  # ✅ Thêm thông tin VRAM target
    },
    tags=["vietnamese", "legal", "gpt-oss", "lora", "16gb-vram"]  # ✅ Thêm tag
)

print("✓ Đã khởi tạo wandb tracking")
print(f"  Project: gpt-oss-legal-vietnamese")
print(f"  Mode: {'Offline' if os.environ.get('WANDB_MODE') == 'offline' else 'Online'}")


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

Lastly we will test it using `reasoning_effort` to `high`

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

### 📝 Lưu ý quan trọng khi finetune GPT-OSS 20B trên 16GB VRAM

**✅ Cấu trúc dữ liệu (GPT-OSS Harmony format):**
- Dataset được format theo GPT-OSS Harmony format với reasoning channels
- Mỗi mẫu gồm: `developer` (system prompt) → `user` (câu hỏi) → `assistant` với 2 channels:
  - `analysis`: Phân tích, lý luận (reasoning process)
  - `final`: Câu trả lời cuối cùng

**✅ Cấu hình tối ưu cho 16GB VRAM:**
- **Model**: `unsloth/gpt-oss-20b-unsloth-bnb-4bit` (BNB 4-bit quantization)
- **LoRA**: r=8, alpha=16, gradient_checkpointing="unsloth" (tiết kiệm 30% VRAM)
- **Batch size**: per_device=1, gradient_accumulation=8 (effective batch=8)
- **Optimizer**: adamw_8bit (8-bit optimizer tiết kiệm VRAM)
- **max_seq_length**: 2048 (đủ cho dữ liệu pháp luật)

**✅ Training:**
- Sử dụng `train_on_responses_only` để chỉ train trên phần assistant (analysis + final)
- Điều chỉnh `max_steps` hoặc `num_train_epochs` theo nhu cầu
- Monitor VRAM usage trong quá trình training

**✅ Reasoning effort (khi inference):**
- `low`: Phân tích nhanh, phù hợp câu hỏi đơn giản
- `medium`: Cân bằng (khuyến nghị cho pháp luật)
- `high`: Phân tích sâu, phù hợp câu hỏi phức tạp


<a name="Data"></a>
### Data Prep

Chúng ta sẽ sử dụng dữ liệu pháp luật Việt Nam từ hai file:
- **qaset_article.json**: Chứa các cặp câu hỏi-trả lời về pháp luật Việt Nam
- **all_chunks_final.json**: Chứa các đoạn văn bản pháp luật gốc

Dataset này giúp mô hình học cách trả lời câu hỏi pháp luật bằng tiếng Việt với khả năng reasoning (phân tích, lập luận) trước khi đưa ra câu trả lời cuối cùng.

In [ ]:
# Load dữ liệu pháp luật Việt Nam từ file JSON
import json
from datasets import Dataset

# Đọc file JSON chứa câu hỏi-trả lời
with open('/home/nhotin/work/LegalBizAI_project/finetune/data_finetunning/qaset_article.json', 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# Đọc file JSON chứa các chunks pháp luật (để tham khảo nếu cần)
with open('/home/nhotin/work/LegalBizAI_project/finetune/data_finetunning/all_chunks_final.json', 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

# Tạo dictionary để tra cứu chunks theo ID
chunks_dict = {chunk['id']: chunk for chunk in chunks_data}

print(f"Đã load {len(qa_data)} câu hỏi-trả lời")
print(f"Đã load {len(chunks_data)} chunks pháp luật")
print(f"\nVí dụ câu hỏi đầu tiên:")
print(f"Q: {qa_data[0]['question'][:100]}...")
print(f"A: {qa_data[0]['answer'][:100]}...")

Chuyển đổi dữ liệu pháp luật Việt Nam sang định dạng GPT-OSS với reasoning channels

In [ ]:
# Chuyển đổi dữ liệu sang format GPT-OSS
def convert_to_gpt_oss_format(qa_item):
    """
    Chuyển đổi một câu hỏi-trả lời pháp luật thành format GPT-OSS
    với reasoning channels (analysis và final)
    
    ✅ Quan trọng: GPT-OSS yêu cầu assistant có 2 channels:
    - analysis: Phân tích, lý luận
    - final: Câu trả lời cuối cùng
    """
    question = qa_item['question']
    answer = qa_item['answer']
    references = qa_item.get('references', [])
    type_question = qa_item.get('type_question', 'query')
    
    # Tạo phần context từ references nếu có
    context_text = ""
    if references:
        ref_texts = []
        for ref in references:
            if len(ref) >= 2:
                ref_texts.append(f"{ref[0]} {ref[1]}")
        if ref_texts:
            context_text = f"\n\nCăn cứ pháp lý: {', '.join(ref_texts)}"
    
    # Tạo phần analysis (reasoning)
    analysis_text = f"""Câu hỏi này liên quan đến pháp luật Việt Nam. Tôi cần phân tích để đưa ra câu trả lời chính xác dựa trên các văn bản quy phạm pháp luật hiện hành.{context_text}

Đây là câu hỏi loại {type_question}, tôi sẽ trả lời dựa trên các quy định cụ thể."""
    
    # Phần final answer
    final_answer = answer
    
    # ✅ Tạo content cho assistant với 2 channels theo format GPT-OSS
    # Format: <|channel|>channel_name<|message|>content
    assistant_content = (
        f"<|channel|>analysis<|message|>{analysis_text}\n"
        f"<|channel|>final<|message|>{final_answer}"
    )
    
    # Tạo messages (KHÔNG dùng channel field, tự format trong content)
    messages = [
        {
            "role": "developer",
            "content": "# Instructions\n\nreasoning language: Vietnamese\n\nBạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành."
        },
        {
            "role": "user",
            "content": question
        },
        {
            "role": "assistant",
            "content": assistant_content  # ✅ Chứa cả 2 channels trong 1 content string
        }
    ]
    
    return {"messages": messages}

# Chuyển đổi tất cả dữ liệu
converted_data = [convert_to_gpt_oss_format(qa) for qa in qa_data]

# Tạo Dataset
dataset = Dataset.from_list(converted_data)

print(f"✓ Đã chuyển đổi {len(dataset)} mẫu dữ liệu sang format GPT-OSS")
print(f"\nVí dụ messages đầu tiên:")
import json
print(json.dumps(dataset[0]['messages'], indent=2, ensure_ascii=False))


Áp dụng chat template và chuẩn bị dữ liệu cho việc training

In [ ]:
# Bước 1: Format messages thành text với chat template
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False
            # ✅ Không cần reasoning_effort vì channels đã có trong content
        ) 
        for convo in convos
    ]
    return {"text": texts}

# Áp dụng formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✓ Dataset đã được format với {len(dataset)} mẫu")
print("\n" + "="*80)
print("Ví dụ text đầu tiên sau khi format:")
print("="*80)
print(dataset[0]['text'][:1500] + "...")  # In 1500 ký tự để thấy channels

# Bước 2: Kiểm tra độ dài của các mẫu dữ liệu
import numpy as np

text_lengths = []
for item in dataset:
    tokens = tokenizer.encode(item['text'])
    text_lengths.append(len(tokens))

print(f"\n{'='*80}")
print(f"Thống kê độ dài token:")
print(f"- Min: {min(text_lengths)}")
print(f"- Max: {max(text_lengths)}")
print(f"- Mean: {np.mean(text_lengths):.1f}")
print(f"- Median: {np.median(text_lengths):.1f}")
print(f"- 95th percentile: {np.percentile(text_lengths, 95):.1f}")

# Cảnh báo nếu có mẫu vượt quá max_seq_length
exceeding = sum(1 for l in text_lengths if l > max_seq_length)
if exceeding > 0:
    print(f"\n⚠️ Cảnh báo: Có {exceeding}/{len(text_lengths)} mẫu ({exceeding/len(text_lengths)*100:.1f}%) vượt quá max_seq_length={max_seq_length}")
    print(f"   Các mẫu này sẽ bị cắt ngắn (truncate). Cân nhắc tăng max_seq_length lên 2048 hoặc 4096.")
else:
    print(f"\n✓ Tốt! Tất cả mẫu đều nằm trong max_seq_length={max_seq_length}")


### 🔍 Kiểm tra format GPT-OSS

GPT-OSS yêu cầu format đặc biệt với **reasoning channels**:

```
<|start|>user<|message|>
[user question]
<|start|>assistant<|channel|>analysis<|message|>
[thinking/reasoning process]
<|channel|>final<|message|>
[final answer]
```

⚠️ **LƯU Ý:** Phải chạy Cell 25 trước để tạo `dataset` với field `text`, sau đó mới chạy cell test bên dưới.

In [ ]:
# 🔍 KIỂM TRA: Verify rằng format đúng với GPT-OSS spec
import re

# Kiểm tra xem dataset đã được tạo chưa
if 'dataset' not in dir() or 'text' not in dataset.column_names:
    print("❌ ERROR: Dataset chưa được format!")
    print("\n👉 Vui lòng chạy Cell 25 trước để format dataset với chat template.")
    raise NameError("Dataset chưa được tạo. Hãy chạy Cell 25 trước.")

# Lấy 1 mẫu text để kiểm tra
sample_text = dataset[0]['text']

print("="*80)
print("🔍 KIỂM TRA FORMAT GPT-OSS")
print("="*80)

# Check 1: Có <|start|>user không?
has_user = '<|start|>user' in sample_text
print(f"\n✓ Có <|start|>user: {has_user}")

# Check 2: Có <|start|>assistant không?
has_assistant = '<|start|>assistant' in sample_text
print(f"✓ Có <|start|>assistant: {has_assistant}")

# Check 3: Có channel analysis không?
has_analysis = '<|channel|>analysis<|message|>' in sample_text
print(f"✓ Có channel 'analysis': {has_analysis}")

# Check 4: Có channel final không?
has_final = '<|channel|>final<|message|>' in sample_text
print(f"✓ Có channel 'final': {has_final}")

if has_user and has_assistant and has_analysis and has_final:
    print("\n" + "="*80)
    print("✅ PASS: Format đúng với GPT-OSS spec!")
    print("="*80)
else:
    print("\n" + "="*80)
    print("❌ FAIL: Format không đúng! Cần kiểm tra lại.")
    print("="*80)

# In ra sample để debug
print("\n" + "="*80)
print("Sample text (first 2000 chars):")
print("="*80)
print(sample_text[:2000])

What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Training mô hình
Bây giờ chúng ta sẽ training mô hình. Bạn có thể điều chỉnh:
- `max_steps`: Số bước training (để test nhanh, dùng 30-60 steps)
- `num_train_epochs`: Số epoch training đầy đủ (bỏ comment và set `max_steps=None` để training hết dataset)
- `per_device_train_batch_size`: Batch size (giảm xuống nếu hết VRAM)

### ⚙️ Disable `torch.compile` (Fix RuntimeError)

Unsloth mặc định dùng `torch.compile` để tăng tốc, nhưng nó conflict với custom loss function. 

→ **Fix:** Disable compile trước khi tạo trainer.

In [ ]:
from trl import SFTConfig, SFTTrainer
import torch

# ✅ CẤU HÌNH TRAINING TỐI ƯU CHO 16GB VRAM:
# - per_device_train_batch_size = 1: Nhỏ nhất để tiết kiệm VRAM
# - gradient_accumulation_steps = 4-8: Tăng effective batch size mà không tốn VRAM
# - optim = "adamw_8bit": 8-bit optimizer tiết kiệm VRAM
# - max_steps: Điều chỉnh theo số lượng dữ liệu

# Tính toán số lượng mẫu để điều chỉnh training
total_samples = len(dataset)
print(f"📊 Thông tin dataset:")
print(f"   Tổng số mẫu training: {total_samples}")

# ✅ Tối ưu batch size cho 16GB VRAM
per_device_batch_size = 1  # ✅ Nhỏ nhất để tiết kiệm VRAM
gradient_accumulation_steps = 8  # ✅ Tăng lên 8 để effective batch size = 8
effective_batch_size = per_device_batch_size * gradient_accumulation_steps

print(f"\n🔧 Cấu hình training cho 16GB VRAM:")
print(f"   per_device_train_batch_size: {per_device_batch_size}")
print(f"   gradient_accumulation_steps: {gradient_accumulation_steps}")
print(f"   Effective batch size: {effective_batch_size}")

# Kiểm tra VRAM trước khi tạo trainer
if torch.cuda.is_available():
    reserved_before = torch.cuda.memory_reserved() / 1024**3
    print(f"   VRAM trước khi tạo trainer: {reserved_before:.2f} GB")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        # ✅ Batch size tối ưu cho 16GB VRAM
        per_device_train_batch_size = per_device_batch_size,  # = 1 để tiết kiệm VRAM
        gradient_accumulation_steps = gradient_accumulation_steps,  # = 8 để effective batch = 8
        
        # ✅ Learning rate và scheduler
        learning_rate = 2e-4,  # Learning rate chuẩn cho LoRA
        lr_scheduler_type = "linear",  # Linear decay
        warmup_steps = 10,  # Warmup ngắn
        
        # ✅ Optimizer tiết kiệm VRAM
        optim = "adamw_8bit",  # ✅ 8-bit optimizer tiết kiệm VRAM
        weight_decay = 0.001,
        
        # ✅ Training steps/epochs
        # Chọn 1 trong 2:
        # Option 1: Training theo steps (nhanh để test)
        max_steps = 100,  # ✅ Điều chỉnh theo nhu cầu (100 cho test, 1000+ cho production)
        # Option 2: Training theo epochs (bỏ comment và set max_steps=None)
        # num_train_epochs = 1,  # Training hết 1 epoch
        
        # ✅ Logging và checkpointing
        logging_steps = 5,  # Log mỗi 5 steps
        logging_first_step = True,  # Log ngay step đầu tiên
        save_strategy = "steps",
        save_steps = 50,  # Lưu checkpoint mỗi 50 steps
        save_total_limit = 2,  # Chỉ giữ 2 checkpoint gần nhất để tiết kiệm disk
        
        # ✅ Wandb tracking
        report_to = "wandb",  # Enable wandb tracking
        run_name = "gpt-oss-20b-16gb-vram",  # Tên run
        
        # ✅ Other settings
        seed = 3407,  # Reproducibility
        output_dir = "outputs",
        
        # ✅ Memory optimization (nếu có)
        dataloader_pin_memory = False,  # Tắt pin memory để tiết kiệm VRAM
        dataloader_num_workers = 0,  # Tắt multiprocessing để tránh memory issues
    ),
)

# Kiểm tra VRAM sau khi tạo trainer
if torch.cuda.is_available():
    reserved_after = torch.cuda.memory_reserved() / 1024**3
    print(f"\n✓ Trainer đã được tạo!")
    print(f"   VRAM sau khi tạo trainer: {reserved_after:.2f} GB")
    print(f"   VRAM tăng thêm: {reserved_after - reserved_before:.2f} GB")
    
    # Cảnh báo nếu VRAM quá cao
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if reserved_after > gpu_memory * 0.9:
        print(f"   ⚠️ Cảnh báo: VRAM sử dụng > 90% ({reserved_after/gpu_memory*100:.1f}%)")
        print(f"   💡 Gợi ý: Giảm max_seq_length hoặc gradient_accumulation_steps nếu gặp OOM")

print(f"\n📋 Tóm tắt cấu hình training:")
print(f"   Effective batch size: {effective_batch_size}")
print(f"   Số steps: {trainer.args.max_steps if trainer.args.max_steps else 'Full epoch'}")
print(f"   Learning rate: {trainer.args.learning_rate}")
print(f"   Optimizer: {trainer.args.optim}")

Chúng ta sử dụng `train_on_responses_only` để chỉ training trên phần trả lời của assistant và bỏ qua phần input của user. Điều này giúp tăng độ chính xác và giảm loss!

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# ✅ Đúng: Chỉ định response_part là điểm bắt đầu của assistant
# GPT-OSS sẽ train trên CẢ 2 channels: analysis + final
gpt_oss_kwargs = dict(
    instruction_part = "<|start|>user<|message|>",
    response_part = "<|start|>assistant"  # ✅ Đúng: Train trên cả analysis + final channels
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)


Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

Now let's print the **masked out** example - you should see only the answer is present:

⚠️ **LƯU Ý:** Cell dưới sẽ báo lỗi nếu bạn chưa chạy Cell 32 (`train_on_responses_only`). 

**Cách fix:**
1. Chạy Cell 30 → Tạo `trainer`
2. Chạy Cell 32 → Apply masking với `train_on_responses_only`
3. Chạy Cell 36 → Xem labels đã được mask


In [ ]:
# Kiểm tra labels có tồn tại không
if 'trainer' not in dir():
    print("❌ ERROR: Trainer chưa được tạo!")
    print("\n👉 Vui lòng chạy Cell 30 trước để tạo trainer.")
    raise NameError("Trainer chưa được tạo. Hãy chạy Cell 30 trước.")

if 'labels' not in trainer.train_dataset[100]:
    print("❌ ERROR: Dataset chưa được mask!")
    print("\n👉 Vui lòng chạy Cell 32 trước để apply train_on_responses_only masking.")
    raise KeyError("Field 'labels' không tồn tại. Hãy chạy Cell 32 (train_on_responses_only) trước.")

# Decode labels để xem phần được train
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Kết thúc wandb tracking
wandb.finish()

print("✓ Đã kết thúc wandb tracking")
print(f"\nĐể xem kết quả training:")
print(f"  1. Nếu dùng online mode: Truy cập https://wandb.ai/your-username/gpt-oss-legal-vietnamese")
print(f"  2. Nếu dùng offline mode: Xem trong thư mục ./wandb/")


### 📊 Wandb Dashboard - Những gì được theo dõi

Wandb tự động track các metrics sau trong quá trình training:

**Training Metrics:**
- `train/loss`: Loss value theo từng step
- `train/learning_rate`: Learning rate schedule
- `train/epoch`: Epoch hiện tại
- `train/global_step`: Tổng số steps đã train

**System Metrics:**
- `system/gpu_utilization`: % sử dụng GPU
- `system/gpu_memory_allocated`: VRAM đã dùng
- `system/cpu_percent`: % CPU usage
- `system/disk_percent`: % disk usage

**Model Configs:**
- Tất cả các hyperparameters từ SFTConfig
- LoRA configs (r, alpha, dropout)
- Model architecture

**Tính năng hữu ích:**
- 📈 Interactive loss curves
- 🔄 So sánh nhiều runs
- 📊 System resource monitoring
- 💾 Tự động save training logs
- 🔗 Share link với team


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference - Test mô hình
Hãy test mô hình đã finetune với câu hỏi pháp luật tiếng Việt!

In [ ]:
# Test với câu hỏi pháp luật tiếng Việt
messages = [
    {"role": "developer", "content": "reasoning language: Vietnamese\n\nBạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam."},
    {"role": "user", "content": "Doanh nghiệp tư nhân cần điều kiện gì để được thành lập?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

<a name="Save"></a>
### Lưu và load mô hình đã finetune
Để lưu model dưới dạng LoRA adapters:
- `save_pretrained`: Lưu local vào thư mục
- `push_to_hub`: Upload lên Hugging Face Hub

**[LƯU Ý]** Hiện tại mô hình finetune chỉ có thể load qua Unsloth. Tính năng export sang vLLM và GGUF đang được phát triển!

In [ ]:
# Lưu mô hình đã finetune
model.save_pretrained("gpt_oss_legal_vietnamese")
tokenizer.save_pretrained("gpt_oss_legal_vietnamese")

print("✓ Đã lưu model vào thư mục: gpt_oss_legal_vietnamese")

# Upload lên Hugging Face Hub (bỏ comment nếu muốn upload)
# model.push_to_hub("your_username/gpt-oss-legal-vietnamese", token = "hf_...")
# tokenizer.push_to_hub("your_username/gpt-oss-legal-vietnamese", token = "hf_...")

To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "gpt_oss_legal_vietnamese", # MODEL ĐÃ FINETUNE
        max_seq_length = 2048,  # Phải giống khi training
        dtype = None,
        load_in_4bit = True,
    )

# Test model đã finetune với câu hỏi pháp luật Việt Nam
messages = [
    {"role": "developer", "content": "reasoning language: Vietnamese\n\nBạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam."},
    {"role": "user", "content": "Hội đồng giải thể doanh nghiệp do Nhà nước nắm giữ 100% vốn điều lệ có được quyền sử dụng con dấu của doanh nghiệp hay không?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

### Lưu dưới dạng float16 hoặc mxfp4

Bạn có thể lưu model dưới dạng `float16` hoặc `mxfp4`:
- `merged_16bit`: Merge LoRA adapters vào model gốc (float16)
- `mxfp4`: Format 4-bit của OpenAI (tiết kiệm VRAM hơn)

Để upload lên Hugging Face, lấy token tại: https://huggingface.co/settings/tokens

In [ ]:
# Lưu và upload dưới dạng mxfp4 4bit format
if False:
    model.save_pretrained_merged("gpt_oss_legal_vietnamese_mxfp4", tokenizer, save_method = "mxfp4")
    print("✓ Đã lưu model mxfp4 vào: gpt_oss_legal_vietnamese_mxfp4")

if False: 
    model.push_to_hub_merged("your_username/gpt-oss-legal-vietnamese-mxfp4", tokenizer, 
                             token = "hf_...", save_method = "mxfp4")

# Lưu và upload dưới dạng float16 (merged)
if False:
    model.save_pretrained_merged("gpt_oss_legal_vietnamese_16bit", tokenizer, save_method = "merged_16bit")
    print("✓ Đã lưu model float16 vào: gpt_oss_legal_vietnamese_16bit")

if False:  # Upload lên Hugging Face Hub
    model.push_to_hub_merged("your_username/gpt-oss-legal-vietnamese-16bit", tokenizer, 
                             save_method = "merged_16bit", token = "hf_...")

## ✅ Hoàn thành! Tổng kết quy trình Finetune GPT-OSS 20B trên 16GB VRAM

### 📋 Quy trình Finetune GPT-OSS 20B cho Pháp luật Việt Nam (Tối ưu 16GB VRAM)

**1️⃣ Chuẩn bị dữ liệu**
- ✅ Load từ `qaset_article.json` và `all_chunks_final.json`
- ✅ Chuyển đổi sang GPT-OSS Harmony format với reasoning channels (analysis + final)
- ✅ Format với chat template và kiểm tra độ dài token

**2️⃣ Cấu hình Model (Tối ưu 16GB VRAM)**
- ✅ **Model**: `unsloth/gpt-oss-20b-unsloth-bnb-4bit` (BNB 4-bit quantization - tiết kiệm VRAM nhất)
- ✅ **LoRA adapters**: r=8, alpha=16, gradient_checkpointing="unsloth" (tiết kiệm 30% VRAM)
- ✅ **max_seq_length**: 2048 (đủ cho dữ liệu pháp luật, không quá lớn)
- ✅ Train chỉ 0.02% parameters (LoRA adapters)

**3️⃣ Training (Tối ưu 16GB VRAM)**
- ✅ **Batch size**: per_device=1, gradient_accumulation=8 (effective batch=8)
- ✅ **Optimizer**: adamw_8bit (8-bit optimizer tiết kiệm VRAM)
- ✅ **SFTTrainer** với `train_on_responses_only` (chỉ train assistant responses)
- ✅ **Wandb tracking** cho monitoring real-time VRAM usage

**4️⃣ Lưu Model**
- **LoRA adapters**: Nhẹ nhất (~8MB), cần base model để inference
- **Merged 16bit**: Full model (~40GB) - không khuyến nghị cho 16GB VRAM
- **MXFP4**: Format 4-bit của OpenAI (tiết kiệm VRAM cho inference)

---

### 🎯 Cấu hình Tối ưu cho 16GB VRAM

| Tham số | Giá trị | Lý do |
|---------|--------|-------|
| Model | `unsloth/gpt-oss-20b-unsloth-bnb-4bit` | BNB 4-bit tiết kiệm VRAM nhất |
| LoRA r | 8 | Rank nhỏ để tiết kiệm VRAM |
| Gradient Checkpointing | "unsloth" | Tiết kiệm 30% VRAM |
| Batch Size | 1 | Nhỏ nhất để tiết kiệm VRAM |
| Gradient Accumulation | 8 | Tăng effective batch mà không tốn VRAM |
| Optimizer | adamw_8bit | 8-bit optimizer tiết kiệm VRAM |
| max_seq_length | 2048 | Đủ cho dữ liệu pháp luật |

---

### 🐛 Common Issues & Fixes

**1. Out of Memory (OOM) Error**
- **Nguyên nhân:** VRAM không đủ
- **Fix:** 
  - Giảm `max_seq_length` xuống 1024 hoặc 512
  - Giảm `gradient_accumulation_steps` xuống 4
  - Đảm bảo đã dùng `unsloth/gpt-oss-20b-unsloth-bnb-4bit` (không phải MXFP4)

**2. KeyError: 'text'**
- **Nguyên nhân:** Dataset chưa được format với chat template
- **Fix:** Chạy Cell 25 để apply `tokenizer.apply_chat_template()`

**3. ZeroDivisionError: All labels are -100**
- **Nguyên nhân:** `response_part` sai, không match với assistant response
- **Fix:** Dùng `response_part = "<|start|>assistant"` (không có `<|message|>`)

**4. KeyError: 'labels'**
- **Nguyên nhân:** Chưa chạy Cell 33 (`train_on_responses_only`)
- **Fix:** Chạy Cell 31 → Cell 33 → Cell 37

---

### 🚀 Next Steps

1. **Monitor training:** Check Wandb dashboard để theo dõi VRAM usage và loss
2. **Evaluate:** Test model với câu hỏi pháp luật mới sau khi training
3. **Deploy:** 
   - Upload LoRA adapters lên HuggingFace (nhẹ, ~8MB)
   - Hoặc merge và export sang MXFP4 format cho inference tiết kiệm VRAM


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
